# CMAPSS IEEE-Grade Experiment Suite (Central + Federated, No Ray)

This notebook is **self-contained** and designed to be **download-and-run** inside your project folder:

```
code_v1/
  dataset/CMAPSSData/   # NASA CMAPSS txt files
  results/
  CMAPSS_IEEE_Experiment_Suite.ipynb
```

✅ **No Ray / no Flower simulator** (so it will not “hang”).  
✅ Produces **many tables (CSV/JSON)** and **many IEEE-style plots (vector PDF)**.

## What you get (aligned with our plan)

### Federated heterogeneity stress tests
- IID vs **condition-skew / severity-skew / quantity-skew / label-skew / stage-skew**
- **Heterogeneity severity sweep** (mild → severe)
- Multi-line metrics + **shaded min–max** across clients

### Robust aggregation + fairness
- **FedAvg vs TrimmedMean vs Median**
- **Worst-client vs average-client** Pareto scatter

### Physics ablation
- Monotonic loss **ON vs OFF**
- **Monotonic violation rate** + **violation magnitude** (per-unit distributions)

### Uncertainty beyond coverage
- Quantile **reliability diagram** (calibration)
- **Sharpness–coverage** Pareto

### Early fault timing
- τ sweep: {10,20,30,40}
- Time-to-failure **recall heatmap** (clients × RUL bins)

### Representation analysis
- Latent **PCA trajectories** (colored by RUL bins / client)
- Cross-client alignment **distance heatmap**

---

> Tip: Start with the **Smoke Test** cell (fast), then run the **Full Suite** cell.


In [ ]:

from __future__ import annotations

import os, json, math, time, random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import matplotlib.pyplot as plt

# ---------------- IEEE-style plotting defaults (as requested) ----------------
def setup_ieee():
    plt.rcParams.update({
        "figure.dpi": 160,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "legend.fontsize": 10,
        "lines.linewidth": 2.0,
    })

def ensure_parent(path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def savefig_pdf(path: Path):
    setup_ieee()
    path = ensure_parent(path)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()

def save_json(path: Path, obj: dict):
    path = ensure_parent(path)
    path.write_text(json.dumps(obj, indent=2))

def save_csv(path: Path, df: pd.DataFrame):
    path = ensure_parent(path)
    df.to_csv(path, index=False)

def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------- Hard-coded project paths ----------------
PROJECT_ROOT = Path(".").resolve()  # run inside code_v1/
DATA_DIR = PROJECT_ROOT / "dataset" / "CMAPSSData"
RES_DIR = PROJECT_ROOT / "results"
RES_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("RES_DIR :", RES_DIR)


## 1) Load CMAPSS FD001/FD004 + correct RUL construction (train/test)

In [ ]:

CMAPSS_COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
FEAT_COLS = ["op1","op2","op3"] + [f"s{i}" for i in range(1,22)]

def read_cmapss_txt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.dropna(axis=1, how="all")
    df = df.iloc[:, :26]
    df.columns = CMAPSS_COLS
    df["unit"] = df["unit"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df

def compute_rul_train(df_train: pd.DataFrame, rul_cap: Optional[int]) -> pd.DataFrame:
    df = df_train.copy()
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df["RUL"] = (max_cycle - df["cycle"]).astype(int)
    if rul_cap is not None:
        df["RUL"] = np.minimum(df["RUL"].values, rul_cap).astype(int)
    return df

def compute_rul_test(df_test: pd.DataFrame, rul_last: np.ndarray, rul_cap: Optional[int]) -> pd.DataFrame:
    df = df_test.copy()
    units = np.sort(df["unit"].unique())
    assert len(units) == len(rul_last), "RUL file length must match number of test units"
    last_rul = {int(u): int(r) for u, r in zip(units, rul_last)}
    last_cycle = df.groupby("unit")["cycle"].transform("max")
    df["RUL"] = (last_cycle - df["cycle"]).astype(int) + df["unit"].map(last_rul).astype(int)
    if rul_cap is not None:
        df["RUL"] = np.minimum(df["RUL"].values, rul_cap).astype(int)
    return df

def load_fd(fd: str, rul_cap: Optional[int]=125) -> Dict[str,pd.DataFrame]:
    assert fd in ["FD001","FD004"], "Use FD001 or FD004"
    train_path = DATA_DIR / f"train_{fd}.txt"
    test_path  = DATA_DIR / f"test_{fd}.txt"
    rul_path   = DATA_DIR / f"RUL_{fd}.txt"
    for p in [train_path, test_path, rul_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}\nPut CMAPSS txt files under {DATA_DIR}")
    train_raw = read_cmapss_txt(train_path)
    test_raw  = read_cmapss_txt(test_path)
    rul_last  = pd.read_csv(rul_path, header=None).values.squeeze()
    train = compute_rul_train(train_raw, rul_cap=rul_cap)
    test  = compute_rul_test(test_raw, rul_last=rul_last, rul_cap=rul_cap)
    return {"train_raw": train_raw, "train": train, "test": test}


## 2) Global (train-based) scaling

In [ ]:

@dataclass
class Scaler:
    mu: np.ndarray
    sigma: np.ndarray

def fit_global_scaler(df_train: pd.DataFrame) -> Scaler:
    X = df_train[FEAT_COLS].values.astype(np.float32)
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    sigma[sigma == 0] = 1.0
    return Scaler(mu=mu, sigma=sigma)

def apply_scaler(df: pd.DataFrame, scaler: Scaler) -> pd.DataFrame:
    out = df.copy()
    X = out[FEAT_COLS].values.astype(np.float32)
    out[FEAT_COLS] = (X - scaler.mu) / scaler.sigma
    return out


## 3) Sliding windows + scarcity protocol + true consecutive-pair sampling for physics loss

In [ ]:

@dataclass
class WindowConfig:
    W: int = 30
    stride: int = 1
    tau: int = 30
    rul_cap: Optional[int] = 125

def make_windows_for_units(df: pd.DataFrame, unit_ids: List[int], cfg: WindowConfig) -> Dict[str, np.ndarray]:
    W, stride = cfg.W, cfg.stride
    X_list, y_rul_list, y_fault_list, unit_list, endc_list = [], [], [], [], []
    for u in unit_ids:
        d = df[df["unit"] == u].sort_values("cycle").reset_index(drop=True)
        feats = d[FEAT_COLS].values.astype(np.float32)
        rul = d["RUL"].values.astype(np.float32)
        cyc = d["cycle"].values.astype(np.int32)
        n = len(d)
        if n < W:
            continue
        for i in range(0, n - W + 1, stride):
            end = i + W - 1
            X_list.append(feats[i:i+W])
            y_rul_list.append(float(rul[end]))
            y_fault_list.append(1.0 if rul[end] <= cfg.tau else 0.0)
            unit_list.append(int(u))
            endc_list.append(int(cyc[end]))
    if len(X_list) == 0:
        return {"X": np.zeros((0,cfg.W,len(FEAT_COLS)), np.float32),
                "y_rul": np.zeros((0,), np.float32),
                "y_fault": np.zeros((0,), np.float32),
                "unit": np.zeros((0,), np.int32),
                "end_cycle": np.zeros((0,), np.int32)}
    return {
        "X": np.stack(X_list, axis=0),
        "y_rul": np.array(y_rul_list, dtype=np.float32),
        "y_fault": np.array(y_fault_list, dtype=np.float32),
        "unit": np.array(unit_list, dtype=np.int32),
        "end_cycle": np.array(endc_list, dtype=np.int32),
    }

def apply_scarcity(arr: Dict[str,np.ndarray], p_pos_percent: float, seed: int = 42) -> Tuple[Dict[str,np.ndarray], int]:
    if p_pos_percent >= 1.0:
        return arr, int((arr["y_fault"]>=0.5).sum())
    rng = np.random.default_rng(seed)
    y = arr["y_fault"]
    pos = np.where(y >= 0.5)[0]
    neg = np.where(y < 0.5)[0]
    keep_pos = int(np.ceil(len(pos) * p_pos_percent))
    sel_pos = rng.choice(pos, size=max(1, keep_pos), replace=False) if len(pos) else np.array([], dtype=int)
    sel = np.concatenate([neg, sel_pos], axis=0)
    sel.sort()
    out = {k: v[sel] for k,v in arr.items()}
    return out, int((out["y_fault"]>=0.5).sum())

def build_pair_map(arr: Dict[str,np.ndarray]) -> Dict[int, Dict[int,int]]:
    pm: Dict[int, Dict[int,int]] = {}
    for i,(u,c) in enumerate(zip(arr["unit"], arr["end_cycle"])):
        pm.setdefault(int(u), {})[int(c)] = int(i)
    return pm

def sample_pairs(pm: Dict[int, Dict[int,int]], units: np.ndarray, endc: np.ndarray, max_pairs: int, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    idxs = np.arange(len(units))
    rng.shuffle(idxs)
    pairs = []
    for i in idxs:
        u = int(units[i]); c = int(endc[i])
        j = pm.get(u, {}).get(c+1, None)
        if j is None:
            continue
        pairs.append((int(i), int(j)))
        if len(pairs) >= max_pairs:
            break
    return np.array(pairs, dtype=np.int64)

class ArrayDataset(Dataset):
    def __init__(self, arr: Dict[str,np.ndarray]):
        self.arr = arr
    def __len__(self): return self.arr["X"].shape[0]
    def __getitem__(self, i: int):
        return (
            torch.from_numpy(self.arr["X"][i]),
            torch.tensor([self.arr["y_fault"][i]], dtype=torch.float32),
            torch.tensor([self.arr["y_rul"][i]], dtype=torch.float32),
            int(self.arr["unit"][i]),
            int(self.arr["end_cycle"][i]),
        )

def make_loader(arr: Dict[str,np.ndarray], batch_size: int, shuffle: bool, imbalance_weighting: bool, seed: int = 42):
    ds = ArrayDataset(arr)
    if shuffle and imbalance_weighting and len(ds) > 0:
        y = arr["y_fault"]
        n_pos = max(1, int((y >= 0.5).sum()))
        n_neg = max(1, int((y < 0.5).sum()))
        w_pos = 0.5 / n_pos
        w_neg = 0.5 / n_neg
        weights = np.where(y >= 0.5, w_pos, w_neg).astype(np.float64)
        sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=True, num_workers=2, pin_memory=True)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle, num_workers=2, pin_memory=True)


## 4) Client splits (IID + Non-IID) and heterogeneity severity levels

In [ ]:

def unit_fail_cycles(train_raw: pd.DataFrame) -> Dict[int,int]:
    return train_raw.groupby("unit")["cycle"].max().to_dict()

def split_units_train_val_test(units: List[int], t_fail_map: Dict[int,int], seed: int = 42) -> Dict[str, List[int]]:
    rng = np.random.default_rng(seed)
    units = np.array(units, dtype=int)
    t = np.array([t_fail_map.get(int(u), 0) for u in units])
    if len(units) >= 10:
        qs = np.quantile(t, [0.2,0.4,0.6,0.8])
        bins = np.digitize(t, qs, right=True)
    else:
        bins = np.zeros_like(t)
    tr, va, te = [], [], []
    for b in np.unique(bins):
        idx = np.where(bins == b)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_tr = max(1, int(round(0.70*n)))
        n_va = max(0, int(round(0.10*n)))
        n_te = n - n_tr - n_va
        if n_te == 0 and n >= 2:
            n_te = 1
            if n_va > 0: n_va -= 1
            else: n_tr -= 1
        tr.extend(units[idx[:n_tr]].tolist())
        va.extend(units[idx[n_tr:n_tr+n_va]].tolist())
        te.extend(units[idx[n_tr+n_va:]].tolist())
    return {"train": tr, "val": va, "test": te}

def make_clients_iid(train_scaled: pd.DataFrame, K: int, seed: int = 42) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    units = np.array(sorted(train_scaled["unit"].unique()), dtype=int)
    rng.shuffle(units)
    chunks = np.array_split(units, K)
    return {i: chunks[i].tolist() for i in range(K)}

def make_clients_severity_skew(train_raw: pd.DataFrame, K: int) -> Dict[int, List[int]]:
    life = train_raw.groupby("unit")["cycle"].max().sort_values()
    units = life.index.values.astype(int)
    chunks = np.array_split(units, K)
    return {i: chunks[i].tolist() for i in range(K)}

def make_clients_quantity_skew(train_raw: pd.DataFrame, K: int, seed: int = 42) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    units = np.array(sorted(train_raw["unit"].unique()), dtype=int)
    rng.shuffle(units)
    weights = np.geomspace(1.0, 0.2, num=K)
    weights = weights / weights.sum()
    counts = (weights * len(units)).astype(int)
    while counts.sum() < len(units): counts[np.argmax(counts)] += 1
    while counts.sum() > len(units): counts[np.argmax(counts)] -= 1
    out, s = {}, 0
    for i in range(K):
        out[i] = units[s:s+counts[i]].tolist()
        s += counts[i]
    return out

def make_clients_condition_skew(train_scaled: pd.DataFrame, K: int, n_clusters: int = 6, seed: int = 42) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    uops = train_scaled.groupby("unit")[["op1","op2","op3"]].mean().reset_index()
    X = uops[["op1","op2","op3"]].values.astype(np.float32)

    cent = X[rng.choice(len(X), size=n_clusters, replace=False)]
    for _ in range(30):
        d2 = ((X[:,None,:] - cent[None,:,:])**2).sum(axis=2)
        lab = d2.argmin(axis=1)
        new_cent = np.vstack([X[lab==j].mean(axis=0) if (lab==j).any() else cent[j] for j in range(n_clusters)])
        if np.allclose(new_cent, cent, atol=1e-4):
            break
        cent = new_cent
    uops["cluster"] = lab

    clusters = []
    for c in range(n_clusters):
        units_c = uops[uops["cluster"]==c]["unit"].astype(int).tolist()
        clusters.append(units_c)
    clusters.sort(key=len, reverse=True)

    out = {i: [] for i in range(K)}
    for idx, units_c in enumerate(clusters):
        out[idx % K].extend(units_c)
    for i in range(K):
        out[i] = sorted(list(set(out[i])))
    return out

def apply_label_skew(client_units: Dict[int,List[int]], train_df_scaled: pd.DataFrame, tau: int, mode: str, seed: int = 42) -> Dict[int,List[int]]:
    rng = np.random.default_rng(seed)
    life = train_df_scaled.groupby("unit")["cycle"].max()
    units_sorted = life.sort_values().index.astype(int).tolist()
    out = {k: list(v) for k,v in client_units.items()}
    if mode == "no_positives_for_some":
        cids = sorted(out.keys())
        rng.shuffle(cids)
        m = max(1, int(0.3*len(cids)))
        healthy_cids = set(cids[:m])
        long_units = units_sorted[::-1]
        ptr = 0
        for cid in healthy_cids:
            take = len(out[cid])
            out[cid] = long_units[ptr:ptr+take]
            ptr += take
    elif mode == "positive_rich_for_some":
        cids = sorted(out.keys())
        rng.shuffle(cids)
        m = max(1, int(0.3*len(cids)))
        rich_cids = set(cids[:m])
        short_units = units_sorted
        ptr = 0
        for cid in rich_cids:
            take = len(out[cid])
            out[cid] = short_units[ptr:ptr+take]
            ptr += take
    else:
        raise ValueError("mode must be 'no_positives_for_some' or 'positive_rich_for_some'")
    return out

def apply_stage_skew(client_units: Dict[int,List[int]], train_raw: pd.DataFrame, severity: float) -> Dict[int,List[int]]:
    out = {k: list(v) for k,v in client_units.items()}
    if severity <= 0:
        return out
    life = train_raw.groupby("unit")["cycle"].max().sort_values()
    units_short = life.index.astype(int).tolist()
    units_long  = units_short[::-1]
    K = len(out)
    pivot = max(1, int(round(K*(0.5*severity))))
    cids = sorted(out.keys())
    for i, cid in enumerate(cids):
        take = len(out[cid])
        if i < pivot:
            out[cid] = units_long[i*take:(i+1)*take]
        elif i >= K-pivot:
            j = i-(K-pivot)
            out[cid] = units_short[j*take:(j+1)*take]
    return out

def severity_levels(level: str) -> Dict[str, float]:
    if level == "mild":
        return {"stage_skew": 0.25, "label_skew": 0.0}
    if level == "medium":
        return {"stage_skew": 0.5, "label_skew": 1.0}
    if level == "severe":
        return {"stage_skew": 0.9, "label_skew": 1.0}
    raise ValueError("level must be mild/medium/severe")

def save_client_mapping(fd: str, K: int, scheme: str, cmap: Dict[int,List[int]], out_path: Path):
    save_json(out_path, {"fd": fd, "K": K, "scheme": scheme, "clients": {str(k): v for k,v in cmap.items()}})


## 5) Model: depthwise-separable TCN + dual-head (fault + quantiles) + health index

In [ ]:

class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.dw = nn.Conv1d(in_ch, in_ch, kernel_size, dilation=dilation, padding=pad, groups=in_ch)
        self.pw = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.bn = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.dw(x)
        y = self.pw(y)
        y = self.bn(y)
        y = F.relu(y)
        y = self.drop(y)
        return y[..., :x.shape[-1]]

class TCNEncoder(nn.Module):
    def __init__(self, n_in, hidden=64, levels=4, kernel=3, dropout=0.1):
        super().__init__()
        layers=[]
        ch=n_in
        for i in range(levels):
            layers.append(DepthwiseSeparableConv1d(ch, hidden, kernel_size=kernel, dilation=2**i, dropout=dropout))
            ch=hidden
        self.net=nn.Sequential(*layers)
    def forward(self, x):
        x = x.transpose(1,2)
        h = self.net(x)
        z = h.mean(dim=-1)
        return z

class DualHeadModel(nn.Module):
    def __init__(self, n_feat, z_dim=64, head=64):
        super().__init__()
        self.encoder = TCNEncoder(n_in=n_feat, hidden=z_dim, levels=4, kernel=3, dropout=0.1)
        self.fault_head = nn.Sequential(nn.Linear(z_dim, head), nn.ReLU(), nn.Linear(head, 1))
        self.q01 = nn.Sequential(nn.Linear(z_dim, head), nn.ReLU(), nn.Linear(head, 1))
        self.d1  = nn.Sequential(nn.Linear(z_dim, head), nn.ReLU(), nn.Linear(head, 1))
        self.d2  = nn.Sequential(nn.Linear(z_dim, head), nn.ReLU(), nn.Linear(head, 1))
        self.health = nn.Sequential(nn.Linear(z_dim, head), nn.ReLU(), nn.Linear(head, 1), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        p = torch.sigmoid(self.fault_head(z))
        q01 = self.q01(z)
        q05 = q01 + F.softplus(self.d1(z))
        q09 = q05 + F.softplus(self.d2(z))
        hi = self.health(z)
        q = torch.cat([q01,q05,q09], dim=1)
        return {"z": z, "p_fault": p, "q": q, "hi": hi}

def get_encoder_state(model: DualHeadModel) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k,v in model.encoder.state_dict().items()}

def set_encoder_state(model: DualHeadModel, enc: Dict[str, torch.Tensor]):
    sd = model.encoder.state_dict()
    model.encoder.load_state_dict({k: enc[k].to(sd[k].device).type_as(sd[k]) for k in sd.keys()})


## 6) Losses + metrics + evaluation

In [ ]:

def focal_loss(p, y, gamma=2.0, alpha=0.75, eps=1e-6):
    p = torch.clamp(p, eps, 1-eps)
    loss = -(alpha*y*(1-p)**gamma*torch.log(p) + (1-alpha)*(1-y)*p**gamma*torch.log(1-p))
    return loss.mean()

def pinball_loss(q, y, a: float):
    e = y - q
    return torch.maximum(a*e, (a-1)*e).mean()

def monotonic_penalty(qi, qj):
    return torch.clamp(qj - qi, min=0.0).mean()

@torch.no_grad()
def compute_metrics_np(y_fault: np.ndarray, p_fault: np.ndarray, y_rul: np.ndarray, q: np.ndarray) -> dict:
    from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, roc_curve
    y = y_fault.astype(np.int32)
    p = p_fault.astype(np.float64)
    out = {}
    out["auprc"] = float(average_precision_score(y, p)) if (np.unique(y).size > 1) else float("nan")
    out["auroc"] = float(roc_auc_score(y, p)) if (np.unique(y).size > 1) else float("nan")
    out["f1@0.5"] = float(f1_score(y, (p>=0.5).astype(int))) if (np.unique(y).size > 1) else 0.0
    fpr, tpr, _ = roc_curve(y, p) if (np.unique(y).size > 1) else (np.array([0.0,1.0]), np.array([0.0,1.0]), None)
    mask = fpr <= 0.01
    out["recall@fpr1%"] = float(tpr[mask].max()) if mask.any() else 0.0

    q05 = q[:,1]
    out["mae"]  = float(np.mean(np.abs(y_rul - q05)))
    out["rmse"] = float(np.sqrt(np.mean((y_rul - q05)**2)))

    q10, q90 = q[:,0], q[:,2]
    out["coverage_q10_q90"] = float(np.mean((y_rul >= q10) & (y_rul <= q90)))
    out["mean_interval_width"] = float(np.mean(q90 - q10))
    return out

@torch.no_grad()
def eval_model(model: DualHeadModel, loader: DataLoader) -> Tuple[float, dict]:
    model.eval()
    tot=0.0; n=0
    Yf=[]; Pf=[]; Yr=[]; Q=[]
    for xb,yf,yr,_,_ in loader:
        xb=xb.to(DEVICE); yf=yf.to(DEVICE); yr=yr.to(DEVICE)
        out=model(xb); p=out["p_fault"]; q=out["q"]
        lf = focal_loss(p, yf)
        lr = pinball_loss(q[:,0:1], yr, 0.1) + pinball_loss(q[:,1:2], yr, 0.5) + pinball_loss(q[:,2:3], yr, 0.9)
        loss = lf + lr
        tot += float(loss.detach().cpu())*xb.size(0); n += xb.size(0)
        Yf.append(yf.detach().cpu().numpy().ravel())
        Pf.append(p.detach().cpu().numpy().ravel())
        Yr.append(yr.detach().cpu().numpy().ravel())
        Q.append(q.detach().cpu().numpy())
    y_fault=np.concatenate(Yf) if Yf else np.zeros((0,))
    p_fault=np.concatenate(Pf) if Pf else np.zeros((0,))
    y_rul=np.concatenate(Yr) if Yr else np.zeros((0,))
    q=np.concatenate(Q) if Q else np.zeros((0,3))
    return tot/max(1,n), compute_metrics_np(y_fault, p_fault, y_rul, q)

@torch.no_grad()
def quantile_reliability(model: DualHeadModel, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    alphas = np.array([0.1, 0.5, 0.9], dtype=np.float32)
    ys=[]; qs=[]
    for xb,_,yr,_,_ in loader:
        xb=xb.to(DEVICE)
        out=model(xb)
        qs.append(out["q"].detach().cpu().numpy())
        ys.append(yr.numpy().ravel())
    y = np.concatenate(ys) if ys else np.zeros((0,))
    q = np.concatenate(qs) if qs else np.zeros((0,3))
    emp = np.array([(y <= q[:,i]).mean() if len(y)>0 else 0.0 for i in range(3)], dtype=np.float32)
    return alphas, emp


## 7) Centralized training + Local client fit (with physics loss)

In [ ]:

@dataclass
class ExpConfig:
    fd: str = "FD004"
    W: int = 30
    stride: int = 1
    tau: int = 30
    p_pos_percent: float = 1.0
    rul_cap: Optional[int] = 125

def train_one_epoch(model: DualHeadModel, loader: DataLoader, pairs: Optional[np.ndarray], opt,
                    lam_fault=1.0, lam_rul=1.0, lam_mono=0.1):
    model.train()
    tot=0.0; n=0
    for xb,yf,yr,_,_ in loader:
        xb=xb.to(DEVICE); yf=yf.to(DEVICE); yr=yr.to(DEVICE)
        out=model(xb); p=out["p_fault"]; q=out["q"]
        lf = focal_loss(p, yf)
        lr = pinball_loss(q[:,0:1], yr, 0.1) + pinball_loss(q[:,1:2], yr, 0.5) + pinball_loss(q[:,2:3], yr, 0.9)
        loss = lam_fault*lf + lam_rul*lr
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        tot += float(loss.detach().cpu())*xb.size(0); n += xb.size(0)

    if pairs is not None and len(pairs) > 0 and lam_mono > 0:
        arr = loader.dataset.arr
        idx = np.random.choice(len(pairs), size=min(1024, len(pairs)), replace=False)
        pp = pairs[idx]
        Xi = torch.from_numpy(arr["X"][pp[:,0]]).to(DEVICE)
        Xj = torch.from_numpy(arr["X"][pp[:,1]]).to(DEVICE)
        qi = model(Xi)["q"][:,1:2]
        qj = model(Xj)["q"][:,1:2]
        lmono = monotonic_penalty(qi, qj)
        opt.zero_grad(set_to_none=True)
        (lam_mono*lmono).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

    return tot/max(1,n)

def train_centralized(exp: ExpConfig, epochs: int = 30, batch_size: int = 64, lr: float = 1e-3,
                      lam_mono: float = 0.1, seed: int = 42):
    seed_all(seed)
    run_id = f"central_{exp.fd}_W{exp.W}_tau{exp.tau}_p{exp.p_pos_percent}_{time.strftime('%Y%m%d_%H%M%S')}"
    OUT = RES_DIR / "central_runs" / run_id
    for p in ["logs","metrics","plots","models","tables","json"]:
        (OUT/p).mkdir(parents=True, exist_ok=True)

    data = load_fd(exp.fd, rul_cap=exp.rul_cap)
    train_raw = data["train"]
    scaler = fit_global_scaler(train_raw)
    train_df = apply_scaler(train_raw, scaler)
    save_json(OUT/"json"/"scaler.json", {"mu": scaler.mu.tolist(), "sigma": scaler.sigma.tolist()})

    units = np.array(sorted(train_df["unit"].unique()), dtype=int)
    rng = np.random.default_rng(seed); rng.shuffle(units)
    n_val = max(1, int(0.2*len(units)))
    val_units = units[:n_val].tolist()
    tr_units  = units[n_val:].tolist()

    wcfg = WindowConfig(W=exp.W, stride=exp.stride, tau=exp.tau, rul_cap=exp.rul_cap)
    tr_arr = make_windows_for_units(train_df, tr_units, wcfg)
    va_arr = make_windows_for_units(train_df, val_units, wcfg)
    tr_arr, pos_kept = apply_scarcity(tr_arr, exp.p_pos_percent, seed=seed)

    pm = build_pair_map(tr_arr)
    pairs = sample_pairs(pm, tr_arr["unit"], tr_arr["end_cycle"], max_pairs=50000, seed=seed)

    tr_ld = make_loader(tr_arr, batch_size=batch_size, shuffle=True,  imbalance_weighting=True, seed=seed)
    va_ld = make_loader(va_arr, batch_size=batch_size, shuffle=False, imbalance_weighting=False, seed=seed)

    model = DualHeadModel(n_feat=len(FEAT_COLS)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    log=[]
    best_mae=1e9
    for ep in range(1, epochs+1):
        tr_loss = train_one_epoch(model, tr_ld, pairs=pairs, opt=opt, lam_mono=lam_mono)
        va_loss, va_m = eval_model(model, va_ld)
        log.append({"epoch": ep, "train_loss": tr_loss, "val_loss": va_loss, **{f"val_{k}":v for k,v in va_m.items()}})
        print(f"[central {exp.fd}] ep {ep:03d} | train_loss {tr_loss:.4f} | val_auprc {va_m['auprc']:.4f} | val_mae {va_m['mae']:.3f}")
        if va_m["mae"] < best_mae:
            best_mae = va_m["mae"]
            torch.save(model.state_dict(), OUT/"models"/"best.pt")

    df = pd.DataFrame(log)
    save_csv(OUT/"logs"/"history.csv", df)

    setup_ieee()
    plt.figure()
    plt.plot(df["epoch"], df["train_loss"], label="train_loss")
    plt.plot(df["epoch"], df["val_loss"], label="val_loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    savefig_pdf(OUT/"plots"/"loss_curve.pdf")

    al, emp = quantile_reliability(model, va_ld)
    setup_ieee()
    plt.figure()
    plt.plot(al, emp, marker="o", label="Empirical")
    plt.plot(al, al, linestyle="--", label="Ideal")
    plt.xlabel("Quantile level α"); plt.ylabel("P(Y ≤ q̂α)")
    plt.legend()
    savefig_pdf(OUT/"plots"/"calibration_reliability.pdf")

    summary = {"config": asdict(exp), "best_val_mae": float(best_mae), "pos_kept": int(pos_kept), "out_dir": str(OUT)}
    save_json(OUT/"metrics"/"summary.json", summary)
    return {"out_dir": str(OUT), "history": df, "summary": summary, "model_state": model.state_dict()}


## 8) Federated Learning simulator (local, no Ray) + experiment drivers

In [ ]:

def aggregate_encoder(states: List[Dict[str, torch.Tensor]], weights: List[float], agg: str, trim_ratio: float) -> Dict[str, torch.Tensor]:
    keys = list(states[0].keys())
    n = len(states)
    if agg == "fedavg":
        w = np.array(weights, dtype=np.float64); w = w / (w.sum() + 1e-12)
        out={}
        for k in keys:
            stack = torch.stack([states[i][k].float() for i in range(n)], dim=0)
            ww = torch.tensor(w, dtype=stack.dtype).view(-1, *([1]*(stack.ndim-1)))
            out[k] = (stack*ww).sum(dim=0)
        return out
    if agg == "median":
        out={}
        for k in keys:
            stack = torch.stack([states[i][k].float() for i in range(n)], dim=0)
            out[k] = stack.median(dim=0).values
        return out
    if agg == "trimmed_mean":
        b = int(np.floor(trim_ratio*n))
        out={}
        for k in keys:
            stack = torch.stack([states[i][k].float() for i in range(n)], dim=0)
            flat = stack.reshape(n, -1).cpu().numpy()
            flat_sorted = np.sort(flat, axis=0)
            trimmed = flat_sorted[b:n-b, :] if (n-2*b)>0 else flat_sorted
            mean = trimmed.mean(axis=0)
            out[k] = torch.from_numpy(mean.reshape(stack.shape[1:])).float()
        return out
    raise ValueError("agg must be fedavg | trimmed_mean | median")

@torch.no_grad()
def monotonic_violation_stats(model: DualHeadModel, arr: Dict[str,np.ndarray], max_units: int = 200) -> dict:
    model.eval()
    pm = build_pair_map(arr)
    units = sorted(pm.keys())[:max_units]
    rates=[]; mags=[]
    for u in units:
        cmap = pm[u]
        cycles = sorted(cmap.keys())
        if len(cycles) < 2:
            continue
        total=0; viol=0; mag_list=[]
        for c in cycles[:-1]:
            j = cmap.get(c+1, None)
            if j is None: 
                continue
            i = cmap[c]
            Xi = torch.from_numpy(arr["X"][i:i+1]).to(DEVICE)
            Xj = torch.from_numpy(arr["X"][j:j+1]).to(DEVICE)
            qi = model(Xi)["q"][:,1].item()
            qj = model(Xj)["q"][:,1].item()
            total += 1
            if qj > qi:
                viol += 1
                mag_list.append(qj - qi)
        if total > 0:
            rates.append(viol/total)
            mags.append(float(np.mean(mag_list)) if len(mag_list)>0 else 0.0)
    return {
        "violation_rate_mean": float(np.mean(rates)) if rates else 0.0,
        "violation_mag_mean":  float(np.mean(mags)) if mags else 0.0,
        "violation_rate_per_unit": rates,
        "violation_mag_per_unit": mags
    }

def run_fl_simulation(exp: ExpConfig, scheme: str="condition-skew", K: int=10, rounds: int=50, local_epochs: int=3,
                      agg: str="fedavg", trim_ratio: float=0.2, imbalance_weighting: bool=True,
                      lam_mono: float=0.1, seed: int=42, batch_size: int=64, lr: float=1e-3,
                      hetero_level: Optional[str]=None, verbose_every: int=1):
    seed_all(seed)
    run_id = f"fl_{exp.fd}_{scheme}_lvl{hetero_level or 'base'}_K{K}_R{rounds}_E{local_epochs}_{agg}_rho{trim_ratio}_mono{lam_mono}_{time.strftime('%Y%m%d_%H%M%S')}"
    OUT = RES_DIR / "fl_runs" / run_id
    for p in ["logs","metrics","plots","models","splits","tables","json"]:
        (OUT/p).mkdir(parents=True, exist_ok=True)

    data = load_fd(exp.fd, rul_cap=exp.rul_cap)
    train_raw = data["train"]
    scaler = fit_global_scaler(train_raw)
    train_df = apply_scaler(train_raw, scaler)
    save_json(OUT/"json"/"scaler.json", {"mu": scaler.mu.tolist(), "sigma": scaler.sigma.tolist()})

    if scheme == "iid":
        cmap = make_clients_iid(train_df, K=K, seed=seed)
    elif scheme == "condition-skew":
        cmap = make_clients_condition_skew(train_df, K=K, n_clusters=6, seed=seed)
    elif scheme == "severity-skew":
        cmap = make_clients_severity_skew(data["train_raw"], K=K)
    elif scheme == "quantity-skew":
        cmap = make_clients_quantity_skew(data["train_raw"], K=K, seed=seed)
    else:
        raise ValueError("scheme must be iid | condition-skew | severity-skew | quantity-skew")

    if hetero_level is not None:
        knobs = severity_levels(hetero_level)
        cmap = apply_stage_skew(cmap, data["train_raw"], severity=knobs["stage_skew"])
        if knobs["label_skew"] > 0:
            cmap = apply_label_skew(cmap, train_df, tau=exp.tau, mode="no_positives_for_some", seed=seed)

    save_client_mapping(exp.fd, K, f"{scheme}_{hetero_level or 'base'}", cmap, OUT/"splits"/"clients.json")

    t_fail_map = unit_fail_cycles(data["train_raw"])
    wcfg = WindowConfig(W=exp.W, stride=exp.stride, tau=exp.tau, rul_cap=exp.rul_cap)

    clients=[]
    val_union=[]
    for cid in range(K):
        units = cmap[cid]
        split = split_units_train_val_test(units, t_fail_map, seed=seed+cid)
        tr_arr = make_windows_for_units(train_df, split["train"], wcfg)
        va_arr = make_windows_for_units(train_df, split["val"],   wcfg)
        te_arr = make_windows_for_units(train_df, split["test"],  wcfg)
        tr_arr, pos_kept = apply_scarcity(tr_arr, exp.p_pos_percent, seed=seed+cid)
        pm = build_pair_map(tr_arr)
        pairs = sample_pairs(pm, tr_arr["unit"], tr_arr["end_cycle"], max_pairs=50000, seed=seed+cid)

        tr_ld = make_loader(tr_arr, batch_size=batch_size, shuffle=True,  imbalance_weighting=imbalance_weighting, seed=seed+cid)
        va_ld = make_loader(va_arr, batch_size=batch_size, shuffle=False, imbalance_weighting=False, seed=seed+cid)
        te_ld = make_loader(te_arr, batch_size=batch_size, shuffle=False, imbalance_weighting=False, seed=seed+cid)

        model = DualHeadModel(n_feat=len(FEAT_COLS)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr)

        clients.append({"cid":cid,"tr_ld":tr_ld,"va_ld":va_ld,"te_ld":te_ld,"pairs":pairs,
                        "pos_kept":int(pos_kept),"n_train":int(tr_arr["X"].shape[0]),"model":model,"opt":opt})
        if va_arr["X"].shape[0] > 0:
            val_union.append(va_arr)

    valU = {k: np.concatenate([a[k] for a in val_union], axis=0) for k in val_union[0].keys()}
    valU_ld = make_loader(valU, batch_size=batch_size, shuffle=False, imbalance_weighting=False, seed=seed)

    global_model = DualHeadModel(n_feat=len(FEAT_COLS)).to(DEVICE)
    global_enc = get_encoder_state(global_model)

    round_rows=[]
    per_client_rows=[]
    for r in range(1, rounds+1):
        t0=time.time()
        for c in clients:
            set_encoder_state(c["model"], global_enc)

        local_states=[]; weights=[]
        for c in clients:
            for _ in range(local_epochs):
                _ = train_one_epoch(c["model"], c["tr_ld"], pairs=c["pairs"], opt=c["opt"], lam_mono=lam_mono)
            local_states.append(get_encoder_state(c["model"]))
            w = float(c["pos_kept"] + 1.0) if imbalance_weighting else float(c["n_train"])
            weights.append(w)

        global_enc = aggregate_encoder(local_states, weights, agg=agg, trim_ratio=trim_ratio)

        val_metrics=[]; test_metrics=[]
        for c in clients:
            set_encoder_state(c["model"], global_enc)
            _, mv = eval_model(c["model"], valU_ld)
            _, mt = eval_model(c["model"], c["te_ld"])
            val_metrics.append(mv); test_metrics.append(mt)
            per_client_rows.append({"round":r,"cid":c["cid"],**mt})

        mean_val = {k: float(np.nanmean([m[k] for m in val_metrics])) for k in val_metrics[0].keys()}
        worst_te = {k: float(np.nanmin([m[k] for m in test_metrics])) for k in test_metrics[0].keys()}
        best_te  = {k: float(np.nanmax([m[k] for m in test_metrics])) for k in test_metrics[0].keys()}
        row={"round":r, **{f"val_mean_{k}":v for k,v in mean_val.items()},
             **{f"test_worst_{k}":v for k,v in worst_te.items()},
             **{f"test_best_{k}":v for k,v in best_te.items()},
             "sec": float(time.time()-t0)}
        round_rows.append(row)
        if (r % verbose_every)==0:
            print(f"Round {r:03d}/{rounds} | val_mae={mean_val['mae']:.3f} | val_auprc={mean_val['auprc']:.3f} | worst_mae={worst_te['mae']:.3f} | {row['sec']:.1f}s")

    df_round=pd.DataFrame(round_rows)
    df_pc=pd.DataFrame(per_client_rows)
    save_csv(OUT/"logs"/"round_summary.csv", df_round)
    save_csv(OUT/"logs"/"per_client_test_metrics.csv", df_pc)

    setup_ieee()
    plt.figure()
    plt.plot(df_round["round"], df_round["val_mean_mae"], label="Val mean MAE")
    plt.plot(df_round["round"], df_round["test_worst_mae"], label="Worst-client MAE")
    plt.fill_between(df_round["round"], df_round["test_worst_mae"], df_round["test_best_mae"], alpha=0.15, label="Client range")
    plt.xlabel("Round"); plt.ylabel("MAE"); plt.legend()
    savefig_pdf(OUT/"plots"/"mae_fairness.pdf")

    torch.save(clients[0]["model"].state_dict(), OUT/"models"/"client0_final.pt")
    torch.save({k:v.cpu() for k,v in global_enc.items()}, OUT/"models"/"global_encoder.pt")

    summary={"config":asdict(exp),"scheme":scheme,"hetero_level":hetero_level,"K":K,"rounds":rounds,
             "local_epochs":local_epochs,"agg":agg,"trim_ratio":trim_ratio,"lam_mono":lam_mono,
             "final": df_round.iloc[-1].to_dict(), "out_dir": str(OUT)}
    save_json(OUT/"metrics"/"summary.json", summary)

    return {"out_dir": str(OUT), "round_summary": df_round, "per_client": df_pc, "summary": summary, "val_union_arr": valU}

# ---- Drivers (heterogeneity sweep, tau sweep, trim sweep, physics ablation) ----
def plot_pareto(df_runs: pd.DataFrame, out_pdf: Path, x_col: str, y_col: str, label_col: str="agg"):
    setup_ieee()
    plt.figure()
    for lab in sorted(df_runs[label_col].unique()):
        d = df_runs[df_runs[label_col]==lab]
        plt.scatter(d[x_col], d[y_col], label=str(lab))
    plt.xlabel(x_col); plt.ylabel(y_col); plt.legend()
    savefig_pdf(out_pdf)

def heterogeneity_sweep(cfg: ExpConfig, K=10, rounds=50, local_epochs=3,
                        agg_list=("fedavg","trimmed_mean","median"),
                        trim_ratio=0.2,
                        base_scheme="condition-skew",
                        levels=("mild","medium","severe"),
                        seed: int = 42):
    runs=[]
    for agg in agg_list:
        out = run_fl_simulation(cfg, scheme="iid", hetero_level=None, K=K, rounds=rounds, local_epochs=local_epochs,
                                agg=agg, trim_ratio=trim_ratio, imbalance_weighting=True, seed=seed, verbose_every=5)
        final = out["summary"]["final"]
        runs.append({"scheme":"iid","level":"base","agg":agg,
                     "auprc": final["val_mean_auprc"], "recall@fpr1%": final["val_mean_recall@fpr1%"],
                     "mae": final["val_mean_mae"], "coverage": final["val_mean_coverage_q10_q90"],
                     "worst_mae": final["test_worst_mae"], "out_dir": out["out_dir"]})
    for lvl in levels:
        for agg in agg_list:
            out = run_fl_simulation(cfg, scheme=base_scheme, hetero_level=lvl, K=K, rounds=rounds, local_epochs=local_epochs,
                                    agg=agg, trim_ratio=trim_ratio, imbalance_weighting=True, seed=seed, verbose_every=5)
            final = out["summary"]["final"]
            runs.append({"scheme":base_scheme,"level":lvl,"agg":agg,
                         "auprc": final["val_mean_auprc"], "recall@fpr1%": final["val_mean_recall@fpr1%"],
                         "mae": final["val_mean_mae"], "coverage": final["val_mean_coverage_q10_q90"],
                         "worst_mae": final["test_worst_mae"], "out_dir": out["out_dir"]})
    df=pd.DataFrame(runs)
    out_csv = RES_DIR/"tables"/f"heterogeneity_sweep_{cfg.fd}_tau{cfg.tau}_{time.strftime('%Y%m%d_%H%M%S')}.csv"
    save_csv(out_csv, df)

    lvl_map = {"base":0,"mild":1,"medium":2,"severe":3}
    df["lvl_idx"] = df["level"].map(lvl_map).fillna(0).astype(int)

    for metric in ["auprc","recall@fpr1%","mae","coverage"]:
        setup_ieee()
        plt.figure()
        for agg in sorted(df["agg"].unique()):
            d = df[df["agg"]==agg].sort_values("lvl_idx")
            plt.plot(d["lvl_idx"], d[metric], marker="o", label=agg)
        plt.xlabel("Heterogeneity level (0=IID,1=mild,2=med,3=severe)")
        plt.ylabel(metric); plt.legend()
        savefig_pdf(RES_DIR/"plots"/f"heterogeneity_{metric}.pdf")

    plot_pareto(df, RES_DIR/"plots"/"pareto_avg_vs_worst_mae.pdf", x_col="mae", y_col="worst_mae", label_col="agg")
    return df

def tau_sweep(fd="FD004", taus=(10,20,30,40), scheme="condition-skew", K=10, rounds=50, local_epochs=3,
              agg="trimmed_mean", trim_ratio=0.2, seed: int = 42):
    rows=[]
    for tau in taus:
        cfg = ExpConfig(fd=fd, W=30, tau=int(tau), p_pos_percent=1.0)
        out = run_fl_simulation(cfg, scheme=scheme, hetero_level="medium", K=K, rounds=rounds, local_epochs=local_epochs,
                                agg=agg, trim_ratio=trim_ratio, imbalance_weighting=True, seed=seed, verbose_every=5)
        final = out["summary"]["final"]
        rows.append({"tau": int(tau),
                     "auprc": final["val_mean_auprc"], "recall@fpr1%": final["val_mean_recall@fpr1%"],
                     "mae": final["val_mean_mae"], "coverage": final["val_mean_coverage_q10_q90"], "out_dir": out["out_dir"]})
    df=pd.DataFrame(rows)
    save_csv(RES_DIR/"tables"/f"tau_sweep_{fd}_{time.strftime('%Y%m%d_%H%M%S')}.csv", df)

    setup_ieee()
    plt.figure()
    plt.plot(df["tau"], df["recall@fpr1%"], marker="o")
    plt.xlabel("Fault horizon τ (cycles)"); plt.ylabel("Recall@FPR=1%")
    savefig_pdf(RES_DIR/"plots"/"tau_sweep_recall_fpr1.pdf")
    return df

def trim_ratio_sweep(cfg: ExpConfig, rhos=(0.0,0.1,0.2,0.3,0.4), scheme="condition-skew", K=10, rounds=50, local_epochs=3, seed: int = 42):
    rows=[]
    for rho in rhos:
        agg="fedavg" if rho==0.0 else "trimmed_mean"
        out = run_fl_simulation(cfg, scheme=scheme, hetero_level="medium", K=K, rounds=rounds, local_epochs=local_epochs,
                                agg=agg, trim_ratio=float(rho), imbalance_weighting=True, seed=seed, verbose_every=5)
        final = out["summary"]["final"]
        rows.append({"rho": float(rho), "agg": agg,
                     "auprc": final["val_mean_auprc"], "recall@fpr1%": final["val_mean_recall@fpr1%"],
                     "mae": final["val_mean_mae"], "coverage": final["val_mean_coverage_q10_q90"],
                     "worst_mae": final["test_worst_mae"], "out_dir": out["out_dir"]})
    df=pd.DataFrame(rows)
    save_csv(RES_DIR/"tables"/f"trim_ratio_sweep_{cfg.fd}_tau{cfg.tau}_{time.strftime('%Y%m%d_%H%M%S')}.csv", df)
    return df

def physics_ablation(cfg: ExpConfig, scheme="condition-skew", K=10, rounds=50, local_epochs=3, agg="trimmed_mean", trim_ratio=0.2, seed: int = 42):
    off = run_fl_simulation(cfg, scheme=scheme, hetero_level="medium", K=K, rounds=rounds, local_epochs=local_epochs,
                            agg=agg, trim_ratio=trim_ratio, imbalance_weighting=True, lam_mono=0.0, seed=seed, verbose_every=5)
    on  = run_fl_simulation(cfg, scheme=scheme, hetero_level="medium", K=K, rounds=rounds, local_epochs=local_epochs,
                            agg=agg, trim_ratio=trim_ratio, imbalance_weighting=True, lam_mono=0.1, seed=seed, verbose_every=5)

    def load_client0(out_dir: str) -> DualHeadModel:
        m = DualHeadModel(n_feat=len(FEAT_COLS)).to(DEVICE)
        m.load_state_dict(torch.load(Path(out_dir)/"models"/"client0_final.pt", map_location=DEVICE))
        return m

    m_off = load_client0(off["out_dir"])
    m_on  = load_client0(on["out_dir"])

    st_off = monotonic_violation_stats(m_off, off["val_union_arr"])
    st_on  = monotonic_violation_stats(m_on,  on["val_union_arr"])

    save_json(RES_DIR/"json"/f"physics_ablation_{cfg.fd}_tau{cfg.tau}_{time.strftime('%Y%m%d_%H%M%S')}.json",
              {"off": st_off, "on": st_on, "off_dir": off["out_dir"], "on_dir": on["out_dir"]})

    setup_ieee()
    plt.figure()
    plt.violinplot([st_off["violation_rate_per_unit"], st_on["violation_rate_per_unit"]], showmeans=True)
    plt.xticks([1,2], ["Mono OFF","Mono ON"]); plt.ylabel("Monotonic violation rate (per-unit)")
    savefig_pdf(RES_DIR/"plots"/"mono_violation_rate_violin.pdf")

    return {"off": off, "on": on, "stats_off": st_off, "stats_on": st_on}


## 9) Smoke test (fast) — run this first

In [ ]:

# cfg = ExpConfig(fd="FD004", W=30, tau=30, p_pos_percent=1.0)

# central_out = train_centralized(cfg, epochs=3, batch_size=64, lr=1e-3, lam_mono=0.1, seed=42)

# fl_out = run_fl_simulation(cfg, scheme="condition-skew", hetero_level="mild",
#                            K=5, rounds=5, local_epochs=1,
#                            agg="fedavg", trim_ratio=0.2,
#                            imbalance_weighting=True,
#                            lam_mono=0.1,
#                            seed=42, batch_size=64, lr=1e-3,
#                            verbose_every=1)

# print("Central out:", central_out["out_dir"])
# print("FL out     :", fl_out["out_dir"])


## 10) Full IEEE-tier suite (heavy)

Uncomment and run.

In [ ]:

cfg = ExpConfig(fd="FD004", W=30, tau=30, p_pos_percent=1.0)

df_hetero = heterogeneity_sweep(cfg, K=10, rounds=50, local_epochs=3,
                               agg_list=("fedavg","trimmed_mean","median"),
                               trim_ratio=0.2,
                               base_scheme="condition-skew",
                               levels=("mild","medium","severe"),
                               seed=42)

df_tau = tau_sweep(fd="FD004", taus=(10,20,30,40), scheme="condition-skew",
                   K=10, rounds=50, local_epochs=3,
                   agg="trimmed_mean", trim_ratio=0.2, seed=42)

df_trim = trim_ratio_sweep(cfg, rhos=(0.0,0.1,0.2,0.3,0.4),
                           scheme="condition-skew", K=10, rounds=50, local_epochs=3, seed=42)

phys = physics_ablation(cfg, scheme="condition-skew", K=10, rounds=50, local_epochs=3,
                        agg="trimmed_mean", trim_ratio=0.2, seed=42)
